In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
class Product:
  def __init__(self, _id, actual_price, average_rating, brand, category, crawled_at, description, discount, images, out_of_stock, pid, product_details, seller, selling_price, sub_category, title, url):
    self._id = _id
    self.actual_price = actual_price
    self.average_rating = average_rating
    self.brand = brand
    self.category = category
    self.crawled_at = crawled_at
    self.description = description
    self.discount = discount
    self.images = images
    self.out_of_stock = out_of_stock
    self.pid = pid
    self.product_details = product_details
    self.seller = seller
    self.selling_price = selling_price
    self.sub_category = sub_category
    self.title = title
    self.url = url

In [37]:
docs_path = 'drive/MyDrive/fashion_products_dataset.json'
import json
with open(docs_path) as fp:
    raw_json = fp.read()
json = json.loads(raw_json)
products = [Product(**product_json) for product_json in json]

In [38]:
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from collections import defaultdict

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [47]:
split_chars = '/.,-'

def split_joined_words(text):
  chars = list(text)
  for i, ch in enumerate(text):
    if ch in split_chars:
      prev_is_letter = i > 0 and text[i - 1].isalpha()
      next_is_letter = i < len(text) - 1 and text[i + 1].isalpha()
      if prev_is_letter or next_is_letter:
        chars[i] = ' '
  return ''.join(chars)

def merge_tshirt(words):
    result = []
    i = 0
    while i < len(words):
        if words[i] == 't' and i + 1 < len(words) and words[i + 1] in ('shirt', 'shirts'):
            result.append('tshirt')
            i += 2
        else:
            result.append('tshirt' if words[i] == 'tshirts' else words[i])
            i += 1
    return result

def build_terms(text: str):
    stemmer = PorterStemmer()
    stop_words = set(stopwords.words("english"))

    text = split_joined_words(text.lower())
    words = merge_tshirt(text.split())       # BEFORE stop words: "t" is an nltk stop word
    words = [w for w in words if w not in stop_words]
    words = [''.join(ch for ch in w if ch.isalnum() or ch in '/%.') for w in words]
    words = [w.strip('./') for w in words]
    words = [w for w in words if w and w not in stop_words]
    return [stemmer.stem(w) for w in words]

In [48]:
def get_product_index(terms, pid):
  index = {}

  for position, term in enumerate(terms):
      try:
          index[term][1].append(position)
      except:
          index[term] = [pid, [position]]
  return index

In [49]:
def preprocess_details(details):
  keep = {'fabric','pattern','sleeve','sleeve type','type','generic name','ideal for','fit', 'brand fit','color','brand color','secondary color','shade','neck type','neck','collar', 'occasion','closure','fly','top closure','hooded','reversible','stretchable','rise','hem', 'faded','distressed','length type','pocket type','pockets','waistband','character', 'fabric details','knit type','weave type','material','outer material','sole material', 'lining','lining material','top fabric','bottom fabric','top type','bottom type', 'coat type','upper pattern'}
  boolean = {'hooded', 'reversible', 'stretchable'}
  product_values = []
  for detail in details:
    for key, value in detail.items():
      key = key.strip().lower()
      value = value.strip().lower()
      if key in keep:
        for part in value.split(','):
          part = part.strip()
          if key in boolean:
              if part == 'yes' and key not in product_values:
                  product_values.append(key)
              continue
          if part.isdigit():
              continue
          if part.startswith(('no ', 'not ')):
              continue
          if part not in product_values:
              product_values.append(part)
  return product_values



In [52]:
def create_indexes(products: list[Product]):
  text_index = defaultdict(list)
  detail_index = defaultdict(list)

  for product in products:

    text_terms = build_terms(product.title + product.description)

    for term, term_info in get_product_index(text_terms, product.pid).items():
      text_index[term].append(term_info)


    current_detail_index = preprocess_details(product.product_details)
    detail_terms = build_terms(' '.join(current_detail_index))

    for term, term_info in get_product_index(detail_terms, product.pid).items():
      detail_index[term].append(term_info)

  return text_index, detail_index

In [53]:
text_index, detail_index = create_indexes(products)


single word tester: blue
text results:  [['TKPFCZ9EJZV2UVRZ', [2]], ['TKPFDABN3GXYPFHE', [3]], ['TKPFCZ9ENWGMX23W', [2]], ['TKPFDACEXAWUHGR7', [3]], ['TKPFD3K6ZMN79MPH', [3]], ['TKPFWBGVGU9FCAYX', [3]], ['TKPFWAG7YFWPMG5Y', [2]], ['TKPFCZ9E2UC3DR3F', [2]], ['SOCFVCCYUDEN2AVH', [85]], ['CTPFVSTDYXRPXGKY', [5]], ['CTPFVZHEKGQ2F67Q', [5]], ['CTPFW3TYNYEDKBV8', [6]], ['CTPFVYARGZYGCKBT', [6]], ['CTPFVPKKTZ2GXGBM', [6]], ['CTPFW3UJCGGVZCZZ', [6]], ['SWTFYGRSFPU6EZJZ', [5]], ['SWTFYHPVHZSVMHYR', [5]], ['TSHFWSZJUQSDZZHG', [6]], ['TSHFYF6KNCXDJRFG', [5]], ['TSHFWSZJDWBTC7PP', [5]], ['TSHFWYV9YZABZYBV', [4]], ['TSHFZ67FFMRPUCAH', [5]], ['TSHFYF6GEZGKCHZA', [6]], ['TSHFWTRVHMN9MG7V', [4]], ['TSHFWTPXPAMQ9YCE', [5]], ['TSHFZ67FRJPUYAWZ', [4]], ['TSHFYFASUQVT8ZVE', [5, 6]], ['TSHFYFASBA2GKAEA', [5]], ['TSHFYFKS6JUSUTUS', [5]], ['TSHFYFASMT4X5T4V', [6]], ['TSHFYFASNXPF75MU', [5]], ['TSHFYFATFYAY698Z', [5]], ['TSHFYFASJWVNGVZG', [4, 6]], ['TSHFYFASAKTVX9B6', [5]], ['TSHFYFASXCZ5MQYX', [6]], ['TSHFY

In [ ]:
search = input('single word tester: ')
search = build_terms(search)[0]

print('text results: ', text_index[search], 'detail results: ', detail_index[search])